In [1]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version

model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       587Mi       9.1Gi       1.0Mi       3.0Gi        11Gi
Swap:             0B          0B          0B
Python 3.12.13


In [2]:
!git clone https://github.com/networkx/networkx.git
%cd networkx
!git checkout networkx-3.2.1
!pip install -e .

Cloning into 'networkx'...
remote: Enumerating objects: 74265, done.
remote: Counting objects: 100% (217/217), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 74265 (delta 138), reused 42 (delta 42), pack-reused 74048 (from 5)
Receiving objects: 100% (74265/74265), 94.11 MiB | 13.70 MiB/s, done.
Resolving deltas: 100% (51602/51602), done.
/content/networkx
Note: switching to 'networkx-3.2.1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 9c1ee6392 Designate 3.2.1 release
Obtaining file:

# Workload

Repository: NetworkX

Commit: networkx-3.2.1

Workload:
Computing betweenness centrality on a large Barabási–Albert graph.

Why meaningful:
Betweenness centrality is widely used in graph analytics and social network analysis. The implementation repeatedly performs breadth-first traversals and dictionary-heavy operations, making it CPU-intensive in Python.

Goal:
Optimize hot-path interpreter overhead without changing algorithm semantics.

In [3]:
import networkx as nx
import pickle
import time

print("Creating graph...")

G = nx.barabasi_albert_graph(2500, 5, seed=42)

print("Graph created")

start = time.perf_counter()

print("Running workload...")

result = nx.betweenness_centrality(G)

end = time.perf_counter()

print(f"Workload time: {end - start:.2f} seconds")

with open("/content/golden.pkl", "wb") as f:
    pickle.dump(result, f)

print("Saved golden output")

Creating graph...
Graph created
Running workload...
Workload time: 34.64 seconds
Saved golden output


In [9]:
!pytest networkx/algorithms/centrality/tests/test_betweenness_centrality.py > /content/baseline_pytest.txt

In [10]:
with open("/content/baseline_pass.txt", "w") as f:
    f.write("PASS")

In [6]:
import time
import statistics

times = []

print("Warmup runs...")

for _ in range(2):
    nx.betweenness_centrality(G)

print("Measured runs...")

for i in range(4):
    start = time.perf_counter()

    nx.betweenness_centrality(G)

    end = time.perf_counter()

    elapsed = end - start

    print(f"Run {i+1}: {elapsed:.2f}s")

    times.append(elapsed)

median = statistics.median(times)

q1 = statistics.quantiles(times, n=4)[0]
q3 = statistics.quantiles(times, n=4)[2]

iqr = q3 - q1

print("Median:", median)
print("IQR:", iqr)

Warmup runs...
Measured runs...
Run 1: 31.43s
Run 2: 31.76s
Run 3: 31.19s
Run 4: 31.89s
Median: 31.590807761000008
IQR: 0.6052618802499978
